### 5.2.3 不确定性传播

我们来研究第1章提及的加工问题.该问题中,若干输入变量服从已知分布.研究目标是通过计算成本高昂的加工仿真器完成输入不确定性传播,得到输出(刀具受力)的分布.

求解这类不确定性传播问题有多种思路.第一种方案:生成大量蒙特卡洛样本$\boldsymbol{x}_i\stackrel{\text{i.i.d.}}{\sim}F,\;i=1,\dots,n$,运行$n$次仿真器,得到输出$\{y_1,\dots,y_n\}$,其中$y_i=h(\boldsymbol{x}_i)$.基于这组输出,可以得到$h(\boldsymbol{x})$的经验分布,或是构建核密度估计.但该方案计算成本很高,仅适用于仿真器运行成本较低的场景.我们可以改用变换低差异序列替代蒙特卡洛样本,降低计算开销;还能进一步采用支撑点方法节约成本——相较于低差异序列,支撑点可以用更少样本达到同等输出分布估计精度.但如果仿真器运行代价极高,则需要先采用高斯过程模型等代理模型对其近似.由于输入分布大概率是非均匀分布,我们可以采用4.6节讨论的投影支撑点或最小能量试验设计构建试验方案.完成代理建模后,再使用大量蒙特卡洛样本求解输出分布.

举个例子,下述函数用于模拟水流通过钻孔的流量$y$(莫里斯等人,1993):

$$y=\frac{2\pi x_3(x_4-x_6)}{\log\left(\frac{x_2}{x_1}\right)\left(1+\frac{2x_7x_3}{\log(x_2/x_1)x_1^2x_8}+\frac{x_3}{x_5}\right)}$$

8个输入变量对应的不确定性分布如表5.1所示.为便于说明,假设该钻孔模型求值成本较高,但仍可以开展代理建模.

>**表5.1**钻孔模型输入变量及其不确定性分布

|输入变量|符号|不确定性分布|
|:--:|:--:|:--:|
|钻孔半径|$x_1$|$\mathcal{\mathcal{N}}(0.1,0.01618^2)$|
|影响半径|$x_2$|$\mathcal{\mathcal{L\mathcal{N}}}(7.71,1.0056^2)$|
|上部透射率|$x_3$|$\mathcal{\mathcal{U}}[63070,115600]$|
|上部水头|$x_4$|$\mathcal{\mathcal{U}}[990,1110]$|
|下部透射率|$x_5$|$\mathcal{\mathcal{U}}[63.1,116]$|
|下部水头|$x_6$|$\mathcal{\mathcal{U}}[700,820]$|
|钻孔长度|$x_7$|$\mathcal{\mathcal{U}}[1120,1680]$|
|钻孔导水系数|$x_8$|$\mathcal{\mathcal{U}}[9855,12045]$|

能量距离基于欧氏距离构建,因此不宜直接在表5.1定义的原始变量空间生成支撑点.我们需要对变量做尺度变换,保证欧氏距离内平方差求和具备合理意义.注意:若$X,X'\stackrel{\text{i.i.d.}}{\sim}F$,则

$$\mathbb{E}[(X-X')^2]=2\mathrm{Var}(X)$$

因此应当对变量尺度调整,使所有变量方差相等,这可以通过常规标准化实现.设$\mu_i$,$\sigma_i$分别为$x_i$的均值与标准差,令

$$z_i=\frac{x_i-\mu_i}{\sigma_i},\;i=1,\dots,p$$

我们在标准化后的空间生成支撑点;得到支撑点后,再通过$x_i=\mu_i+\sigma_iz_i$映射回原始变量空间.

针对钻孔模型,我们首先生成100000组蒙特卡洛样本.钻孔模型求值难度较低,因此可以直接在100000组样本上计算模型输出,并通过核密度估计得到输出分布,对应图5.8中黑色虚线.下面采用支撑点方法实现不确定性传播:首先对100000组蒙特卡洛样本标准化,借助R语言程序包`support`(马克,2018)得到$n=100$个支撑点$\{\boldsymbol{z}_j\}_{j=1}^{100}$;再通过$x_{ji}=\mu_i+\sigma_iz_{ji}$将支撑点还原至原始尺度.在这100个支撑点上求解钻孔模型,基于得到的$y_i$构建核密度估计,对应图中绿色实线.可以看到,该结果能够很好逼近由100000组蒙特卡洛样本得到的"真实"密度.作为对照,图中同时给出100组蒙特卡洛样本得到的密度曲线(红色虚线).不难看出,支撑点得到的密度估计与真实密度更加接近.

> **图5.8** 采用100个蒙特卡洛样本(红色虚线)与100个支撑点(绿色实线)得到的钻孔函数输出分布.真实输出分布由黑色虚线给出

In [2]:
#图5.8

options(repr.plot.width=10,repr.plot.height=10)
f=function(x)2*pi*x[3]*(x[4]-x[6])/(log(x[2]/x[1])*(1+2*x[7]*x[3]/(log(x[2]/x[1])*x[1]^2*x[8])+x[3]/x[5]))
p=8;N=100000;n=100
lower=c(0.05,100,63070,990,63.1,700,1120,9855)
upper=c(0.15,50000,115600,1110,116,820,1680,12045)
set.seed(1)
X=matrix(0,nrow=N,ncol=p)
X[,1]=0.1+0.01618*rnorm(N)
X[,2]=exp(7.71+1.0056*rnorm(N))
X[,3:p]=matrix(runif(N*6),nrow=N)
X[,3:p]=sweep(X[,3:p],2,upper[3:p]-lower[3:p],"*")
X[,3:p]=sweep(X[,3:p],2,lower[3:p],"+")
true=apply(X,1,f)
plot(density(true),lty=3,lwd=4,xlim=c(0,200),cex.axis=2,xlab="y",ylab="密度",cex.lab=2,main="输出分布",cex.main=3)

x=X[sample(1:N,n),]
mc=apply(x,1,f)
lines(density(mc),col=2,lty=2,lwd=4)

mu=apply(X,2,mean)
sigma=apply(X,2,sd)
Z=X|>
sweep(2,mu,"-")|>
sweep(2,sigma,"/")
library(support)
D=sp(n,p,dist.samp=Z)$sp
D=D|>
sweep(2,sigma,"*")|>
sweep(2,mu,"+")
y=apply(D,1,f)
lines(density(y),col=3,lwd=4)
legend(100,.015,legend=c("真实值","蒙特卡洛样本","支撑点"),col=c(1,2,3),lwd=4,lty=c(3,2,1),bty="n",cex=2)